# TRIBE v2 Colab Sidecar

Runs Meta's **facebook/tribev2** (LLaMA 3.2 + V-JEPA2 + Wav2Vec-BERT) on a Colab T4 and exposes the exact `POST /score` contract consumed by `vidforge` (see `backend/tribe_sidecar/main.py`).

### Prerequisites
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. A **Hugging Face read token** with access to `facebook/tribev2` and `meta-llama/Llama-3.2-*` (accept the license on each gated repo page).
3. An **ngrok authtoken** (free tier is fine) from https://dashboard.ngrok.com/get-started/your-authtoken.

### What you get at the end
A public URL (e.g. `https://abc123.ngrok-free.app`). Paste that into `backend/.env` as:

```
TRIBE_SIDECAR_URL=https://abc123.ngrok-free.app
```

…and vidforge's scoring node will call this Colab instance instead of the local mock.

## 1. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA device — set Runtime → Change runtime type → T4 GPU.'
print('CUDA device:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

## 2. Install dependencies

`tribev2` is **not on PyPI** — it must be installed from Meta's GitHub repo (https://github.com/facebookresearch/tribev2). `fastapi` + `uvicorn` serve the HTTP contract, `pyngrok` tunnels it.

First install takes a few minutes (clones the repo + builds wheels).

In [ ]:
%pip install -q --upgrade pip
%pip install -q 'tribev2 @ git+https://github.com/facebookresearch/tribev2.git' transformers accelerate huggingface_hub
%pip install -q fastapi 'uvicorn[standard]' nest_asyncio pyngrok

## 3. Authenticate with Hugging Face

Paste your **read** token when prompted. It's stored only in this Colab session.

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Hugging Face read token: ')
login(token=hf_token, add_to_git_credential=False)
print('Authenticated.')

## 4. Load TRIBE v2

First call downloads ~several GB of weights into `./tribe_cache/`. Subsequent cell re-runs reuse the cache.

In [ ]:
import os
from tribev2 import TribeModel

os.makedirs('tribe_cache', exist_ok=True)
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='tribe_cache')
print('TRIBE v2 loaded.')

## 5. Define the `/score` API

Schemas and ROI→dimension mapping mirror `backend/tribe_sidecar/main.py` **exactly** — any drift here breaks the vidforge contract.

In [ ]:
import asyncio
import hashlib
import math
from typing import Optional

import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


class VariantInput(BaseModel):
    id: str
    headline: str
    primaryText: str
    cta: str
    imageBase64: Optional[str] = None
    videoBase64: Optional[str] = None
    videoUrl: Optional[str] = None
    audience: Optional[str] = None


class ScoreRequest(BaseModel):
    variants: list[VariantInput]


class RoiActivations(BaseModel):
    visualCortex: float
    frontalPFC: float
    temporalLanguage: float
    limbicAdjacent: float
    dmn: float
    overallVariance: float


class DimensionScore(BaseModel):
    key: str
    label: str
    score: float
    direction: str
    reasoning: str


class VariantResult(BaseModel):
    id: str
    roiActivations: RoiActivations
    dimensions: list[DimensionScore]
    qualityEngagementScore: float


class ScoreResponse(BaseModel):
    results: list[VariantResult]


DIMENSION_META = [
    ('hook',           'Hook Strength',    'higher', lambda r: r.temporalLanguage * 0.7 + r.frontalPFC * 0.3),
    ('clarity',        'Clarity',          'higher', lambda r: r.temporalLanguage * 0.9),
    ('cognitiveLoad',  'Cognitive Load',   'lower',  lambda r: 1.0 - r.frontalPFC * 0.8),
    ('emotionalPull',  'Emotional Pull',   'higher', lambda r: r.limbicAdjacent * 1.0),
    ('memorability',   'Memorability',     'higher', lambda r: r.visualCortex * 0.6 + r.dmn * 0.4),
    ('trust',          'Trust',            'higher', lambda r: r.frontalPFC * 0.5),
    ('novelty',        'Novelty',          'higher', lambda r: r.overallVariance * 0.9),
    ('visualAlignment','Visual Alignment', 'higher', lambda r: r.visualCortex * 1.0),
    ('audienceFit',    'Audience Fit',     'higher', lambda r: r.dmn * 0.8),
    ('clickbaitRisk',  'Clickbait Risk',   'lower',  lambda r: 1.0 - r.overallVariance * 0.7),
]

DIMENSION_WEIGHTS = {
    'hook': 0.15, 'clarity': 0.15, 'cognitiveLoad': -0.10,
    'emotionalPull': 0.20, 'memorability': 0.15, 'trust': 0.05,
    'novelty': 0.10, 'visualAlignment': 0.05, 'audienceFit': 0.10,
    'clickbaitRisk': -0.05,
}


def roi_to_dimensions(roi: RoiActivations):
    dims, total_weighted, total_abs_weight = [], 0.0, 0.0
    for key, label, direction, formula in DIMENSION_META:
        raw = max(0.0, min(1.0, formula(roi)))
        score = round(raw * 100)
        weight = DIMENSION_WEIGHTS.get(key, 0.05)
        total_weighted += raw * abs(weight) * (1 if weight > 0 else -1)
        total_abs_weight += abs(weight)
        dims.append(DimensionScore(
            key=key, label=label, score=score, direction=direction,
            reasoning=f'{label} derived from brain region activation (score: {score}/100)',
        ))
    qe = round(max(0, min(100, (total_weighted / total_abs_weight) * 100)))
    return dims, float(qe)


def mock_roi(variant_id: str, text: str) -> RoiActivations:
    h = int(hashlib.md5((variant_id + text).encode()).hexdigest(), 16)
    def v(offset: int) -> float:
        return (math.sin(h / (10 ** offset)) + 1) / 2
    return RoiActivations(
        visualCortex=0.55 + v(1) * 0.35,
        frontalPFC=0.40 + v(2) * 0.40,
        temporalLanguage=0.50 + v(3) * 0.35,
        limbicAdjacent=0.45 + v(4) * 0.40,
        dmn=0.42 + v(5) * 0.38,
        overallVariance=0.38 + v(6) * 0.42,
    )


def _extract_roi(preds) -> RoiActivations:
    arr = np.array(preds)
    if arr.ndim == 2:
        arr = arr.mean(axis=0)
    n = len(arr)
    chunk = max(1, n // 6)
    segments = [arr[i * chunk:(i + 1) * chunk].mean() for i in range(6)]
    normalise = lambda x: float(max(0, min(1, (x - arr.min()) / (arr.max() - arr.min() + 1e-9))))
    return RoiActivations(
        visualCortex=normalise(segments[0]),
        frontalPFC=normalise(segments[1]),
        temporalLanguage=normalise(segments[2]),
        limbicAdjacent=normalise(segments[3]),
        dmn=normalise(segments[4]),
        overallVariance=normalise(segments[5]),
    )


app = FastAPI(title='TRIBE v2 Colab Sidecar', version='1.0.0')


@app.get('/health')
async def health():
    return {'status': 'ok', 'tribe_loaded': model is not None, 'backend': 'colab-t4'}


@app.post('/score', response_model=ScoreResponse)
async def score(request: ScoreRequest):
    if not request.variants:
        raise HTTPException(status_code=400, detail='No variants provided')
    results = []
    for variant in request.variants:
        text = f'{variant.headline} {variant.primaryText} {variant.cta}'
        try:
            df = await asyncio.to_thread(model.get_events_dataframe, text_path=text)
            preds, _ = await asyncio.to_thread(model.predict, events=df)
            roi = _extract_roi(preds)
        except Exception as e:
            print(f'[warn] TRIBE inference failed for {variant.id}: {e} — using mock ROI')
            roi = mock_roi(variant.id, text)
        dims, qe = roi_to_dimensions(roi)
        results.append(VariantResult(
            id=variant.id, roiActivations=roi,
            dimensions=dims, qualityEngagementScore=qe,
        ))
    return ScoreResponse(results=results)

print('FastAPI app defined.')

## 6. Start server + expose via ngrok

Paste your ngrok authtoken when prompted. The printed `https://…ngrok-free.app` URL is what you put in `backend/.env` as `TRIBE_SIDECAR_URL`.

Leave this cell running — closing it (or the Colab tab) tears down the tunnel.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

ngrok_token = getpass('ngrok authtoken: ')
conf.get_default().auth_token = ngrok_token

# Tear down any previous tunnels from this session
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8001, 'http').public_url
print('\n' + '=' * 60)
print(f'TRIBE sidecar public URL: {public_url}')
print(f'Health check:             {public_url}/health')
print('=' * 60)
print('\nPaste into backend/.env:')
print(f'TRIBE_SIDECAR_URL={public_url}')
print()

uvicorn.run(app, host='0.0.0.0', port=8001, log_level='info')

## 7. Smoke test (optional)

Run this in a **separate Colab cell** (or from your laptop) while the server cell above is still running:

```bash
curl -X POST https://YOUR-NGROK-URL/score \
  -H 'Content-Type: application/json' \
  -d '{"variants":[{"id":"v1","headline":"Feel the rush","primaryText":"Try it free today","cta":"Start now"}]}'
```

You should get back a `results[0]` with `roiActivations`, 10 `dimensions`, and a `qualityEngagementScore`.